# Unicycle dynamics

Four networks, one recipe. Train/val is the first 75% / last 25% of the nine training collections. Test is figure 8, diagonal lawnmower, random trajectory, slalom (held out whole).

Train on **10 s** windows. The checkpoint is the lowest **10 s** xyyaw on val. Commands are commanded `(v, ω)`.

Checkpoints: `runs/unicycle/<family>/seed_k/{best.pt,last.pt}`. Training resumes `last.pt` if it exists. After each family finishes, the notebook plots **30 s** open-loop rollouts on each test collection starting at **t = 0, 30, 60 s**.

In [1]:
SEED = 0  # change to start or resume another copy; eval uses seeds 0–4
DEVICE = "cpu"

## Dataset

Left vs right commanded turns, and the `(v, ω)` the robot was asked to run.

In [2]:
from unicycle.experiments.data import load_split, turn_counts
from unicycle.experiments.viz import plot_command_heatmap, plot_split_balance

train, val, test = load_split()
for name, logs in (("train", train), ("val", val), ("test", test)):
    print(name, turn_counts(logs))
plot_split_balance(train, val, test)
plot_command_heatmap(train, title="train")
plot_command_heatmap(test, title="test")

train {'left': 5571, 'right': 4891, 'straight': 17084, 'steps': 27546}
val {'left': 2173, 'right': 1391, 'straight': 5616, 'steps': 9180}
test {'left': 3993, 'right': 3951, 'straight': 8777, 'steps': 16721}


PosixPath('/Users/antony/Desktop/GTech_FA26/capstone/learned-dynamics-and-symmetry/unicycle/experiments/visualizations/command_vw_test.png')

## Visualizations

Writes `experiments/visualizations/stills` and `gifs` for every train, val, and test collection.

In [ ]:
from unicycle.experiments.viz import write_collections

write_collections(train, val, test)

In [ ]:
# imports

from quadrotor.geometry import train_device
from unicycle.experiments.models import NAIVE, QUOTIENT, QUOTIENT_AUG, QUOTIENT_SYM
from unicycle.experiments.train.lib import train_family
from unicycle.experiments.viz import plot_test_rollouts

device = train_device(DEVICE)

## NaiveUnicycle

MLP on the 7-D world state. No SE(2) quotient.

In [ ]:
model = train_family(NAIVE, SEED, device=device)
plot_test_rollouts(model, test, NAIVE)

## QuotientUnicycleNet

Same MLP, input is `(v, ω, v_cmd, ω_cmd)`.

In [ ]:
model = train_family(QUOTIENT, SEED, device=device)
plot_test_rollouts(model, test, QUOTIENT)

## QuotientUnicycleNet + data augmentation

Same quotient net. Training data is concatenated with its left–right mirror.

In [ ]:
model = train_family(QUOTIENT_AUG, SEED, device=device)
plot_test_rollouts(model, test, QUOTIENT_AUG)

## SymmetricQuotientMLP

Same quotient MLP. Evaluates both Z₂ elements (identity and left–right: wheels swap, ω flips, v stays) and averages the increment. This is the predicted step, not a residual on the analytic plant.

In [3]:
model = train_family(QUOTIENT_SYM, SEED, device=device)
plot_test_rollouts(model, test, QUOTIENT_SYM)

NameError: name 'train_family' is not defined